In [6]:
# ==============================================================================
# Funções Canônicas e Corpus de 8 Documentos (Aula 08 - Tema 3)
# ==============================================================================

# 1. Corpus de 8 documentos da aula
docs <- c(
  d1 = "recuperacao de informacao ordena documentos por relevancia",
  d2 = "o modelo de espaco vetorial usa tf-idf e similaridade de cosseno",
  d3 = "o bm25 e um modelo probabilistico de recuperacao de texto",
  d4 = "relevancia e a correspondencia entre necessidade e documento",
  d5 = "precisao e revocacao avaliam a qualidade do ranking",
  d6 = "consultas curtas dificultam a recuperacao precisa",
  d7 = "aprendizado de maquina pode aprender a ordenar documentos",
  d8 = "o indice invertido mapeia termos aos documentos que os contem"
)

# 2. Configurações canônicas (o "de" fica no vocabulário)
cfg <- list(
  minusculas = TRUE,
  remover_pontuacao = TRUE,
  stopwords = NULL
)

# 3. Funções de preparação e indexação
preparar <- function(txt, cfg) {
  if (isTRUE(cfg$minusculas)) txt <- tolower(txt)
  if (isTRUE(cfg$remover_pontuacao)) txt <- gsub("[^[:alnum:] ]", " ", txt)
  toks <- unlist(strsplit(txt, "\\s+"))
  toks <- toks[toks != ""]
  if (!is.null(cfg$stopwords)) toks <- toks[!toks %in% cfg$stopwords]
  toks
}

montar <- function(docs, cfg) {
  tokens <- lapply(docs, function(d) preparar(d, cfg))
  vocab  <- sort(unique(unlist(tokens)))
  N      <- length(docs)
  df     <- sapply(vocab, function(t) sum(sapply(tokens, function(toks) t %in% toks)))

  # IDF probabilístico do BM25
  idf_bm25 <- log((N - df + 0.5) / (df + 0.5) + 1)

  list(tokens = tokens, vocab = vocab, N = N, df = df, idf_bm25 = idf_bm25)
}

# 4. Construir o índice
ix <- montar(docs, cfg)

# 5. Função de Cobertura (Módulo 1)
cobertura <- function(qt, dt, idf) {
  qt <- unique(qt[qt %in% names(idf)])         # termos distintos da consulta no vocabulário
  if (!length(qt)) return(0)                   # evita divisão por zero
  presentes <- qt[qt %in% dt]                  # termos que o documento contém
  sum(idf[presentes]) / sum(idf[qt])           # fração do IDF coberta
}

# 6. Teste da cobertura para "modelo de recuperacao"
qt <- preparar("modelo de recuperacao", cfg)
cv <- sapply(ix$tokens, function(dt) cobertura(qt, dt, ix$idf_bm25))

cat("Valores de cobertura para 'modelo de recuperacao':\n")
print(round(cv, 3))

Valores de cobertura para 'modelo de recuperacao':
   d1    d2    d3    d4    d5    d6    d7    d8 
0.561 0.676 1.000 0.000 0.000 0.324 0.237 0.000 


In [ ]:
# 1. Função BM25 canônica
ranking_bm25 <- function(txt, ix, cfg, k1 = 1.2, b = 0.75) {
  qt <- preparar(txt, cfg)
  qt <- qt[qt %in% ix$vocab]
  if (!length(qt)) return(setNames(numeric(ix$N), names(ix$tokens)))

  doc_lens <- sapply(ix$tokens, length)
  avgdl    <- mean(doc_lens)

  scores <- sapply(names(ix$tokens), function(id) {
    dt <- ix$tokens[[id]]
    tf <- table(dt)
    termos_comuns <- intersect(qt, names(tf))
    if (!length(termos_comuns)) return(0)

    sum(sapply(termos_comuns, function(t) {
      f <- as.numeric(tf[t])
      idf <- ix$idf_bm25[t]
      K <- k1 * (1 - b + b * (length(dt) / avgdl))
      idf * (f * (k1 + 1)) / (f + K)
    }))
  })
  sort(scores, decreasing = TRUE)
}

# 2. Reordenador por Cobertura no Top-k
reordenar_cobertura <- function(r_bm25, txt, ix, cfg, k = 3, peso_cob = 1.0) {
  if (k <= 1 || length(r_bm25) == 0) return(r_bm25)

  candidatos <- names(r_bm25)[1:min(k, length(r_bm25))]
  resto      <- names(r_bm25)[(min(k, length(r_bm25)) + 1):length(r_bm25)]

  qt <- preparar(txt, cfg)
  # Calcula cobertura apenas para os candidatos do top-k
  cobs <- sapply(candidatos, function(d) cobertura(qt, ix$tokens[[d]], ix$idf_bm25))

  # Normaliza BM25 dos candidatos para escala para combinar
  b_cand <- r_bm25[candidatos]
  b_norm <- if (max(b_cand) > min(b_cand)) (b_cand - min(b_cand)) / (max(b_cand) - min(b_cand)) else rep(1, length(b_cand))

  score_final <- b_norm + peso_cob * cobs
  novo_top <- names(sort(score_final, decreasing = TRUE))

  c(novo_top, resto)
}

# 3. Teste prático na consulta "modelo de recuperacao"
rb <- ranking_bm25("modelo de recuperacao", ix, cfg)
r_reord <- reordenar_cobertura(rb, "modelo de recuperacao", ix, cfg, k = 3)

cat("--- Top-3 BM25 Original ---\n")
print(names(rb)[1:3])

cat("\n--- Top-3 Reordenado com Cobertura (k=3) ---\n")
print(r_reord[1:3])

--- Top-3 BM25 Original ---
[1] "d3" "d2" "d1"

--- Top-3 Reordenado com Cobertura (k=3) ---
[1] "d3" "d2" "d1"


In [ ]:
# Teste de consulta com empates
q_dupla <- "informacao recuperacao"
qt_d    <- preparar(q_dupla, cfg)

# Cobertura nos 8 documentos
cv_d <- sapply(ix$tokens, function(dt) cobertura(qt_d, dt, ix$idf_bm25))

cat("--- Cobertura para 'informacao recuperacao' ---\n")
print(round(cv_d, 3))

cat("\nTabela de frequências dos valores de cobertura:\n")
print(table(round(cv_d, 3)))

--- Cobertura para 'informacao recuperacao' ---
   d1    d2    d3    d4    d5    d6    d7    d8 
1.000 0.000 0.345 0.000 0.000 0.345 0.000 0.000 

Tabela de frequências dos valores de cobertura:

    0 0.345     1 
    5     2     1 


In [ ]:
# ==============================================================================
# Módulo 4: Reordenação por Cobertura no Corpus do Grupo (Baixada Santista)
# ==============================================================================

# 1. Carregamento dos dados
corpus_df <- read.csv("corpus_formatado.csv", stringsAsFactors = FALSE)
qrels_df  <- read.csv("qrelsTOT.csv", stringsAsFactors = FALSE)
nec_df    <- read.csv("necessidades.csv", stringsAsFactors = FALSE)

# 2. Montar índice do corpus do grupo
docs_grupo <- setNames(corpus_df$texto, as.character(corpus_df$id))
ix_grupo   <- montar(docs_grupo, cfg)

# Consultas
consultas_grupo <- setNames(nec_df$texto_consulta, nec_df$consulta)

# 3. Métrica de Avaliação: Average Precision (AP)
calc_ap <- function(ranking_docs, qid, qrels) {
  gabarito <- as.character(qrels[qrels$consulta == qid & qrels$grau > 0, "documento"])
  R <- length(gabarito)
  if (R == 0) return(0)

  recuperados_rel <- ranking_docs %in% gabarito
  if (!any(recuperados_rel)) return(0)

  posicoes <- which(recuperados_rel)
  precisoes <- seq_along(posicoes) / posicoes
  sum(precisoes) / R
}

# 4. Avaliar BM25 + Reordenação por Cobertura para k in {0, 5, 10, 20}
dev_q <- paste0("q0", 1:7)
dev_q <- dev_q[dev_q %in% names(consultas_grupo)]

testar_grade <- function(queries) {
  ks <- c(k0_bm25 = 0, k5 = 5, k10 = 10, k20 = 20)
  sapply(ks, function(k_val) {
    aps <- sapply(queries, function(qid) {
      txt <- consultas_grupo[qid]
      if (is.na(txt) || txt == "") return(0)

      rb  <- ranking_bm25(txt, ix_grupo, cfg)
      if (k_val > 0) {
        rf <- reordenar_cobertura(rb, txt, ix_grupo, cfg, k = k_val)
      } else {
        rf <- names(rb)
      }
      calc_ap(rf, qid, qrels_df)
    })
    mean(aps)
  })
}

cat("--- MAP no Conjunto de Desenvolvimento (q01 a q07) ---\n")
map_dev <- testar_grade(dev_q)
print(round(map_dev, 4))

--- MAP no Conjunto de Desenvolvimento (q01 a q07) ---
k0_bm25      k5     k10     k20 
 0.3157  0.3157  0.3167  0.3130 


In [ ]:
# ==============================================================================
# Avaliação Final no Conjunto de Teste (q08 a q10)
# ==============================================================================

teste_q <- c("q08", "q09", "q10")
teste_q <- teste_q[teste_q %in% names(consultas_grupo)]

cat("--- MAP no Conjunto de Teste (q08 a q10) ---\n")
map_teste_bm25 <- mean(sapply(teste_q, function(qid) {
  txt <- consultas_grupo[qid]
  rb  <- ranking_bm25(txt, ix_grupo, cfg)
  calc_ap(names(rb), qid, qrels_df)
}))

map_teste_cob10 <- mean(sapply(teste_q, function(qid) {
  txt <- consultas_grupo[qid]
  rb  <- ranking_bm25(txt, ix_grupo, cfg)
  rf  <- reordenar_cobertura(rb, txt, ix_grupo, cfg, k = 10)
  calc_ap(rf, qid, qrels_df)
}))

cat("BM25 puro (k=0):", round(map_teste_bm25, 4), "\n")
cat("Cobertura (k=10):", round(map_teste_cob10, 4), "\n")

--- MAP no Conjunto de Teste (q08 a q10) ---
BM25 puro (k=0): 0.3919 
Cobertura (k=10): 0.4018 
